In [ ]:
import os


import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import timm
import copy
import math
import time
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

import flatbuffers
import litert_torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from ai_edge_litert import schema_py_generated as schema
from ai_edge_litert.interpreter import Interpreter, OpResolverType
from ai_edge_quantizer import quantizer, recipe
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.cuda.manual_seed_all(seed)
       torch.backends.cudnn.deterministic = False
       torch.backends.cudnn.benchmark = True

set_seed(SEED)

In [ ]:
!pwd

# Setup

In [ ]:
PATH = "/home/jovyan/work/UFSC/butterflies_austria/Dataset_train_test_split/butterflies-austria"

EPOCHS = 40
PATIENCE_ES = 8

BATCH_SIZE = 64


LR = 1e-3
MIN_LR = 1e-7
GRAD_CLIP = 1.0

IMG_SIZE = (224, 224)
MODEL_SAVE_PATH = "best__model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

# Dataset

In [ ]:
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Lambda(lambda image: image.convert("RGB")),
    transforms.RandomHorizontalFlip(p=0.1),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Lambda(lambda image: image.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

In [ ]:
train_dir = os.path.join(PATH, "train")
val_dir = os.path.join(PATH, "val")
test_dir = os.path.join(PATH, "test")

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transforms)

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

if val_dataset.class_to_idx != class_to_idx:
    raise ValueError("Training and validation classes are not aligned")

if test_dataset.class_to_idx != class_to_idx:
    raise ValueError("Training and test classes are not aligned")

pin_memory = DEVICE.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=pin_memory,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=pin_memory,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=pin_memory,
    persistent_workers=True,
)

print(f"Classes: {len(class_names)}")
print(
    f"Train: {len(train_dataset)} | "
    f"Val: {len(val_dataset)} | "
    f"Test: {len(test_dataset)}"
)

In [ ]:
train_targets = np.asarray(train_dataset.targets)
train_counts = np.bincount(train_targets, minlength=len(class_names))

print("Training distribution:")
for name, count in zip(class_names, train_counts):
    print(f"  {name:30s}: {count}")

plt.figure(figsize=(12, 5))
plt.bar(class_names, train_counts)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Number of samples")
plt.title("Class distribution (training)")
plt.tight_layout()
plt.show()

class_weights = len(train_targets) / (
    len(class_names) * train_counts
)

class_weights = np.sqrt(class_weights)
class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE,
)
print(f"Class weights: {class_weights.tolist()}")

In [ ]:
def plot_samples(dataset, class_names, num_samples=9):
    cols = 3
    rows = math.ceil(num_samples / cols)
    plt.figure(figsize=(10, 10))
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    indices = random.sample(range(len(dataset)), num_samples)
    for i, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        img_np = (img_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        
        ax = plt.subplot(rows, cols, i + 1)
        plt.imshow(img_np)
        plt.title(class_names[label])
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

plot_samples(train_dataset, class_names)
plot_samples(test_dataset, class_names)
plot_samples(val_dataset, class_names)

# Model

In [ ]:
model = timm.create_model(
    "mobilenetv2_100",
    pretrained=True,
    num_classes=len(class_names),
    drop_rate=0.3,
).to(DEVICE)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

# Train

In [ ]:
# Fine-tuning: classifier novo (aleatorio) precisa de LR maior que o backbone pre-treinado
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
head_params     = [p for n, p in model.named_parameters() if 'classifier' in n]

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": LR / 10},
        {"params": head_params, "lr": LR},
    ],
    weight_decay=1e-4,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05,
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR / 10, LR],
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    div_factor=100,
    final_div_factor=100,
)

# Treinamento
best_acc = 0.0
patience_counter = 0
best_model_wts = copy.deepcopy(model.state_dict())

train_losses, val_losses = [], []
train_accs, val_accs = [], []

In [ ]:
print("Starting training...")

for epoch in tqdm(range(EPOCHS), desc="Training Progress"):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    model.eval()
    val_loss_running = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss_running += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = val_loss_running / len(val_dataset)
    val_acc = correct_val / total_val
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    tqdm.write(
        f"[{epoch+1:03d}/{EPOCHS}] "
        f"train_loss={epoch_loss:.4f} train_acc={epoch_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE_ES:
            tqdm.write(f"Early stopping triggered at epoch {epoch + 1}.")
            break

model.load_state_dict(best_model_wts)
print(f"Best validation accuracy: {best_acc:.4f}")

# Test

In [ ]:
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"Final test accuracy: {test_correct / test_total:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
def predict_and_plot(model, dataset, classes, num_samples=10):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    model.eval()
    indices = random.sample(range(len(dataset)), num_samples)

    cols = 5
    rows = math.ceil(num_samples / cols)

    plt.figure(figsize=(15, 4 * rows))
    for i, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        img_input = img_tensor.unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(img_input)
            pred = output.argmax(dim=1).item()

        img_np = (img_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img_np)

        color = 'green' if pred == label else 'red'
        plt.title(f"True: {classes[label]}\nPredicted: {classes[pred]}", color=color, fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

predict_and_plot(model, test_dataset, class_names, num_samples=20)

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

labels = range(len(class_names))

print(
    classification_report(
        all_labels,
        all_preds,
        labels=labels,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(all_labels, all_preds, labels=labels)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Exportação e validação para ESP32-S3

A exportação abaixo separa a adaptação específica da arquitetura das validações genéricas do artefato TFLite. Para outro modelo, ajuste `EXPORT_ADAPTER`, os shapes e a lista de operadores suportados pelo firmware. O modelo final só é aprovado quando usa buffers inline, schema compatível, quantização INT8 completa e resultados consistentes nos dois resolvers.


In [ ]:
EXPORT_DIR = Path("tflite_models")
EXPORT_DIR.mkdir(exist_ok=True)

NUM_CALIB_SAMPLES = 100
MAX_RESOLVER_ACCURACY_DELTA = 0.01

ESP32_INPUT_SHAPE = (1, 3, 224, 224)
ESP32_OUTPUT_SHAPE = (1, len(class_names))
ESP32_SUPPORTED_OP_VERSIONS = {
    "ADD": frozenset({1}),
    "AVERAGE_POOL_2D": frozenset({1}),
    "CONV_2D": frozenset({1}),
    "DEPTHWISE_CONV_2D": frozenset({1}),
    "FULLY_CONNECTED": frozenset({1}),
    "PAD": frozenset({1}),
    "TRANSPOSE": frozenset({1}),
}
ESP32_SUPPORTED_OPS = frozenset(ESP32_SUPPORTED_OP_VERSIONS)

FP32_PATH = EXPORT_DIR / "model_fp32.tflite"
INT8A16_PATH = EXPORT_DIR / "model_int8a16.tflite"
INT8_PATH = EXPORT_DIR / "model_int8.tflite"


In [ ]:
calibration_dataset = datasets.ImageFolder(train_dir, transform=val_test_transforms)

In [ ]:
class MobileNetV2FixedAvgPool(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, inputs):
        outputs = self.backbone.forward_features(inputs)
        spatial_shape = tuple(outputs.shape[-2:])
        if spatial_shape != (7, 7):
            raise ValueError(f"Feature map esperado: 7x7; recebido: {spatial_shape}")

        outputs = F.avg_pool2d(outputs, kernel_size=(7, 7), stride=1)
        outputs = torch.flatten(outputs, 1)
        outputs = F.dropout(
            outputs,
            p=self.backbone.drop_rate,
            training=self.backbone.training,
        )
        return self.backbone.classifier(outputs)


def mobilenet_v2_export_adapter(source_model):
    return MobileNetV2FixedAvgPool(source_model)


def prepare_export_model(source_model, inputs, adapter=None):
    baseline = copy.deepcopy(source_model).cpu().eval()
    candidate = adapter(baseline).eval() if adapter else baseline

    with torch.no_grad():
        expected = baseline(inputs[0])
        actual = candidate(inputs[0])

    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    max_error = float(torch.max(torch.abs(expected - actual)))
    print(f"Erro absoluto máximo do adaptador: {max_error:.10f}")
    return candidate


sample_image = test_dataset[0][0]
sample_inputs = (sample_image.unsqueeze(0).cpu(),)
EXPORT_ADAPTER = mobilenet_v2_export_adapter
export_model = prepare_export_model(model, sample_inputs, EXPORT_ADAPTER)

print(sample_inputs[0].shape)


In [ ]:
def inline_external_buffers(source_path, target_path):
    content = source_path.read_bytes()
    model = schema.Model.GetRootAsModel(content, 0)
    model_object = schema.ModelT.InitFromObj(model)
    converted = 0

    for index, target_buffer in enumerate(model_object.buffers):
        source_buffer = model.Buffers(index)
        offset = source_buffer.Offset()
        size = source_buffer.Size()
        if not offset or not size:
            continue

        end = offset + size
        if end > len(content):
            raise ValueError(f"Buffer {index} fora do arquivo: {offset}:{end}")

        target_buffer.data = bytearray(content[offset:end])
        target_buffer.offset = 0
        target_buffer.size = 0
        converted += 1

    builder = flatbuffers.Builder(len(content))
    model_offset = model_object.Pack(builder)
    builder.Finish(model_offset, file_identifier=b"TFL3")
    target_path.write_bytes(builder.Output())
    return converted


def export_with_inline_buffers(exporter, output_path):
    external_path = output_path.with_name(
        f"{output_path.stem}_external{output_path.suffix}"
    )
    inline_path = output_path.with_name(f".{output_path.name}.tmp")
    external_path.unlink(missing_ok=True)
    inline_path.unlink(missing_ok=True)

    try:
        exporter(external_path)
        converted = inline_external_buffers(external_path, inline_path)
        os.replace(inline_path, output_path)
    finally:
        external_path.unlink(missing_ok=True)
        inline_path.unlink(missing_ok=True)

    print(f"{output_path.name}: {converted} buffers incorporados")


edge_model = litert_torch.convert(export_model, sample_inputs)
export_with_inline_buffers(
    lambda output_path: edge_model.export(str(output_path)),
    FP32_PATH,
)

print(f"FP32: {FP32_PATH.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
def create_calibration_data(model_path, dataset, num_samples=200, seed=42):
    interpreter = Interpreter(model_path=str(model_path))
    signatures = interpreter.get_signature_list()

    if len(signatures) != 1:
        raise ValueError(f"Expected one signature, found: {list(signatures)}")

    signature_name, signature = next(iter(signatures.items()))
    input_names = signature["inputs"]

    if len(input_names) != 1:
        raise ValueError(f"Expected one input, found: {input_names}")

    input_name = input_names[0]

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:min(num_samples, len(dataset))].tolist()

    samples = []

    for index in indices:
        image = dataset[index][0]
        samples.append({input_name: image.unsqueeze(0).cpu().numpy().astype(np.float32)})

    return {signature_name: samples}

calibration_data = create_calibration_data(FP32_PATH, calibration_dataset, NUM_CALIB_SAMPLES)

In [ ]:
def quantize_static(fp32_path, output_path, calibration_data, quantization_recipe):
    qt = quantizer.Quantizer(str(fp32_path))
    qt.load_quantization_recipe(quantization_recipe())
    calibration_result = qt.calibrate(calibration_data)
    result = qt.quantize(calibration_result)

    export_with_inline_buffers(
        lambda external_path: result.export_model(
            str(external_path),
            overwrite=True,
        ),
        output_path,
    )

    print(f"{output_path.name}: {output_path.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
quantize_static(FP32_PATH, INT8A16_PATH, calibration_data, recipe.static_wi8_ai16)

In [ ]:
quantize_static(FP32_PATH, INT8_PATH, calibration_data, recipe.static_wi8_ai8)

In [ ]:
def create_interpreter(model_path, resolver_type):
    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_op_resolver_type=resolver_type,
    )
    interpreter.allocate_tensors()
    return interpreter


def operator_names(interpreter):
    return [operation["op_name"] for operation in interpreter._get_ops_details()]


def external_buffer_indices(model):
    return [
        index
        for index in range(model.BuffersLength())
        if model.Buffers(index).Offset() or model.Buffers(index).Size()
    ]


def operator_versions(model):
    names = {
        value: name
        for name, value in vars(schema.BuiltinOperator).items()
        if name.isupper() and isinstance(value, int)
    }
    return [
        (
            names.get(
                model.OperatorCodes(index).BuiltinCode(),
                str(model.OperatorCodes(index).BuiltinCode()),
            ),
            model.OperatorCodes(index).Version(),
        )
        for index in range(model.OperatorCodesLength())
    ]


def validate_tflite_model(model_path, require_full_int8):
    content = model_path.read_bytes()
    if len(content) < 8 or content[4:8] != b"TFL3":
        raise RuntimeError(f"{model_path.name}: identificador TFL3 inválido")

    model = schema.Model.GetRootAsModel(content, 0)
    if model.Version() != 3:
        raise RuntimeError(
            f"{model_path.name}: schema {model.Version()} não suportado"
        )

    external_buffers = external_buffer_indices(model)
    if external_buffers:
        raise RuntimeError(
            f"{model_path.name}: buffers externos {external_buffers}"
        )

    interpreter = create_interpreter(
        model_path,
        OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    inputs = interpreter.get_input_details()
    outputs = interpreter.get_output_details()
    if len(inputs) != 1 or len(outputs) != 1:
        raise RuntimeError(
            f"{model_path.name}: esperado uma entrada e uma saída"
        )

    input_info = inputs[0]
    output_info = outputs[0]
    input_shape = tuple(int(value) for value in input_info["shape"])
    output_shape = tuple(int(value) for value in output_info["shape"])
    if input_shape != ESP32_INPUT_SHAPE:
        raise RuntimeError(
            f"{model_path.name}: entrada {input_shape}, esperado {ESP32_INPUT_SHAPE}"
        )
    if output_shape != ESP32_OUTPUT_SHAPE:
        raise RuntimeError(
            f"{model_path.name}: saída {output_shape}, esperado {ESP32_OUTPUT_SHAPE}"
        )

    operators = operator_names(interpreter)
    if "SUM" in operators:
        raise RuntimeError(f"{model_path.name}: operador SUM não permitido")

    unsupported = sorted(set(operators) - ESP32_SUPPORTED_OPS)
    if unsupported:
        raise RuntimeError(
            f"{model_path.name}: operadores não registrados no firmware: {unsupported}"
        )

    versions = operator_versions(model)
    unsupported_versions = [
        (name, version)
        for name, version in versions
        if version not in ESP32_SUPPORTED_OP_VERSIONS.get(name, frozenset())
    ]
    if unsupported_versions:
        raise RuntimeError(
            f"{model_path.name}: versões de operadores não validadas: "
            f"{unsupported_versions}"
        )

    if require_full_int8:
        if input_info["dtype"] != np.int8 or output_info["dtype"] != np.int8:
            raise RuntimeError(f"{model_path.name}: entrada e saída devem ser int8")

        float_tensors = [
            detail["name"]
            for detail in interpreter.get_tensor_details()
            if detail["dtype"] in (np.float16, np.float32, np.float64)
        ]
        if float_tensors:
            raise RuntimeError(
                f"{model_path.name}: tensores não quantizados: {float_tensors}"
            )

        for tensor_info in (input_info, output_info):
            scales = tensor_info["quantization_parameters"]["scales"]
            if len(scales) != 1 or float(scales[0]) <= 0:
                raise RuntimeError(
                    f"{model_path.name}: quantização de I/O inválida"
                )

    print(model_path.name)
    print(f"Schema: {model.Version()}")
    print(f"Input:  {input_shape} {input_info['dtype']}")
    print(f"Output: {output_shape} {output_info['dtype']}")
    print(f"Operators: {operators}")
    print(f"Operator versions: {versions}")
    print(f"Size: {model_path.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
validate_tflite_model(FP32_PATH, require_full_int8=False)
validate_tflite_model(INT8A16_PATH, require_full_int8=False)
validate_tflite_model(INT8_PATH, require_full_int8=True)


In [ ]:
def evaluate_tflite(
    model_path,
    data_loader,
    class_names,
    resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
):
    interpreter = create_interpreter(model_path, resolver_type)

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_index = input_details["index"]
    output_index = output_details["index"]
    input_dtype = input_details["dtype"]
    output_dtype = output_details["dtype"]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    all_labels = []
    all_predictions = []

    for images, labels in data_loader:
        images = images.cpu().numpy()

        for image, label in zip(images, labels.numpy()):
            inputs = np.expand_dims(image, axis=0)

            if np.issubdtype(input_dtype, np.integer):
                if input_scale <= 0:
                    raise ValueError(f"Quantização inválida em {model_path}")
                inputs = np.round(inputs / input_scale + input_zero_point)
                inputs = np.clip(
                    inputs,
                    np.iinfo(input_dtype).min,
                    np.iinfo(input_dtype).max,
                ).astype(input_dtype)
            else:
                inputs = inputs.astype(input_dtype)

            interpreter.set_tensor(input_index, inputs)
            interpreter.invoke()
            output = interpreter.get_tensor(output_index)

            if np.issubdtype(output_dtype, np.integer) and output_scale > 0:
                output = (
                    output.astype(np.float32) - output_zero_point
                ) * output_scale

            all_labels.append(int(label))
            all_predictions.append(int(np.argmax(output, axis=1)[0]))

    accuracy = accuracy_score(all_labels, all_predictions)
    print(f"{model_path.name} | {resolver_type.name}: {accuracy:.4f}")
    print(
        classification_report(
            all_labels,
            all_predictions,
            target_names=class_names,
            digits=4,
            zero_division=0,
        )
    )
    return accuracy, all_labels, all_predictions


In [ ]:
fp32_results = evaluate_tflite(
    FP32_PATH,
    test_loader,
    class_names,
)


In [ ]:
int8_builtin_results = evaluate_tflite(
    INT8_PATH,
    test_loader,
    class_names,
    OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
)
int8_reference_results = evaluate_tflite(
    INT8_PATH,
    test_loader,
    class_names,
    OpResolverType.BUILTIN_REF,
)

accuracy_delta = abs(int8_builtin_results[0] - int8_reference_results[0])
prediction_agreement = float(np.mean(
    np.asarray(int8_builtin_results[2]) == np.asarray(int8_reference_results[2])
))

print(f"Diferença de acurácia: {accuracy_delta:.4f}")
print(f"Concordância das predições: {prediction_agreement:.4f}")

if accuracy_delta > MAX_RESOLVER_ACCURACY_DELTA:
    raise RuntimeError(
        "Divergência excessiva entre os resolvers; modelo rejeitado"
    )

int8_results = int8_builtin_results


In [ ]:
int8a16_results = evaluate_tflite(
    INT8A16_PATH,
    test_loader,
    class_names,
)


In [ ]:
def print_tflite_io_types(model_path):
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    print(f"\n{model_path.name}")
    print(f"inference_input_type:  {input_details['dtype']}")
    print(f"inference_output_type: {output_details['dtype']}")
    print(f"input quantization:     {input_details['quantization']}")
    print(f"output quantization:    {output_details['quantization']}")

print_tflite_io_types(FP32_PATH)
print_tflite_io_types(INT8A16_PATH)
print_tflite_io_types(INT8_PATH)

In [ ]:
print("Modelo INT8 aprovado para teste no ESP32-S3")
print(f"Artefato: {INT8_PATH.resolve()}")
